In [57]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage,HumanMessage
import operator

In [87]:
generator_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
evaluator_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite") 


In [59]:
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation:Literal["approved", "needs_improvement"]
    feedback:str
    iteration:int
    max_iteration:int
    tweet_history:Annotated[list[str], operator.add]
    feedback_history:Annotated[list[str], operator.add]

In [81]:
class TweetEvaluation(BaseModel):
    evaluation : Literal["approved", "needs_improvement"] = Field(description="Final evaluation result")
    feedback : str = Field(description="Constructive feedback for the tweet")

In [82]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [62]:
def generate_tweet(state:TweetState):
    # prompt 
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
    Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

    Rules:
    - Do NOT use question-answer format.
    - Max 280 characters.
    - Use observational humor, irony, sarcasm, or cultural references.
    - Think in meme logic, punchlines, or relatable takes.
    - Use simple, day to day english
    """)]
    
    response = generator_llm.invoke(messages).content
    return {'tweet':response, 'tweet_history':[response]}


def evaluate_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
        HumanMessage(content=f"""
        Evaluate the following tweet:

        Tweet: "{state['tweet']}"

        Use the criteria below to evaluate the tweet:

        1. Originality – Is this fresh, or have you seen it a hundred times before?  
        2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
        3. Punchiness – Is it short, sharp, and scroll-stopping?  
        4. Virality Potential – Would people retweet or share it?  
        5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

        Auto-reject if:
        - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
        - It exceeds 280 characters
        - It reads like a traditional setup-punchline joke
        - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

        ### Respond ONLY in structured format:
        - evaluation: "approved" or "needs_improvement"  
        - feedback: One paragraph explaining the strengths and weaknesses 
        """)
        ]
    
    response = structured_evaluator_llm.invoke(messages)
    return {"evaluation":response.evaluation, "feedback":response.feedback, 'feedback_history':[response.feedback]}

def optimize_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
        Improve the tweet based on this feedback:
        "{state['feedback']}"

        Topic: "{state['topic']}"
        Original Tweet:
        {state['tweet']}

        Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
        """)]
    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration'] + 1
    return {"tweet":response, "iteration":iteration}

In [63]:
def route_evaluation(state:TweetState):
    if state['evaluation'] == "approved" or  state['iteration'] > state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [83]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved':END, 'needs_improvement':'optimize'})
graph.add_edge('optimize', 'evaluate')


workflow = graph.compile()



In [84]:
workflow.get_graph().print_ascii()

          +-----------+             
          | __start__ |             
          +-----------+             
                 *                  
                 *                  
                 *                  
           +----------+             
           | generate |             
           +----------+             
                 *                  
                 *                  
                 *                  
           +----------+             
           | evaluate |             
           +----------+             
          ...         ..            
         .              ..          
       ..                 .         
+---------+           +----------+  
| __end__ |           | optimize |  
+---------+           +----------+  


In [85]:
initial_state = {
    'topic':"rq",
    'iteration':1,
    'max_iteration':3
    }
result = workflow.invoke(initial_state)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 1.860058254s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_d

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 59.441867537s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 59
}
]

In [86]:
result

{'topic': "ksdaflsda'ksa",
 'tweet': 'My autocorrect just tried to tell me "ksdaflsda\'ksa" is a word. Pretty sure that\'s the sound my brain makes when I try to remember where I put my keys. 🔑 #AutocorrectFail #BrainFog #IsThisASpiritAnimal',
 'evaluation': 'approved',
 'feedback': 'The tweet is relatable and humorous, effectively using the autocorrect fail to describe brain fog. The hashtags are relevant and add to the discoverability. The format is concise and engaging, making it highly shareable.',
 'iteration': 1,
 'max_iteration': 3,
 'tweet_history': ['My autocorrect just tried to tell me "ksdaflsda\'ksa" is a word. Pretty sure that\'s the sound my brain makes when I try to remember where I put my keys. 🔑 #AutocorrectFail #BrainFog #IsThisASpiritAnimal'],
 'feedback_history': ['The tweet is relatable and humorous, effectively using the autocorrect fail to describe brain fog. The hashtags are relevant and add to the discoverability. The format is concise and engaging, making it h